# CITADEL Single-Notebook Experiment Runner

This notebook is the one-stop runner for the CITADEL journal-extension workspace. It executes the reproducible sample pipeline, reproduces the EXACT baseline, runs the CITADEL/TCAD ablation grid, attaches hardware-cost estimates, exports FPGA/RTL golden vectors, and shows how future RTL synthesis results can be merged back into the paper tables.

The default settings run the deterministic smoke workload so the notebook finishes on a laptop. For final TCAD numbers, change `DATA_MODE` to `"real"`, point `REAL_DATA_ROOT` at an immutable telemetry snapshot, and set `TCAD_PRESET` to `"full"`.

In [1]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "exact").is_dir():
            return candidate
    raise RuntimeError(f"Could not find CITADEL repo root from {start}")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository: {REPO_ROOT}")
print(f"Python: {platform.python_version()} on {platform.platform()}")

Repository: /Users/hsiaopingni/Documents/New project/EXACT-TCAD
Python: 3.13.5 on macOS-26.3.1-arm64-arm-64bit-Mach-O


## 1. Reproducibility Configuration

The notebook uses the same deterministic settings as the command-line runner:

- seed: `123`
- numerical threads: `1`
- deterministic sample-data generator
- repo-relative paths
- run manifests with package versions, git commit, input hashes, and output hashes

For the strictest cross-machine comparison, start Jupyter itself with `PYTHONHASHSEED=123`. The notebook still sets the environment variable for subprocesses and downstream tools.

In [2]:
SEED = 123
THREADS = 1
SAMPLE_ROWS = 600

# Laptop-safe default. Switch to "real" when your immutable telemetry snapshot is ready.
DATA_MODE = "sample"  # "sample" or "real"
REAL_DATA_ROOT = REPO_ROOT / "data" / "telemetry" / "processed" / "<snapshot_id>"
TCAD_PRESET = "smoke"  # "smoke" for laptop/debug, "full" for journal-scale sweeps
RUN_REPEAT_CHECK = True

RESULTS_ROOT = REPO_ROOT / "results" / "notebook_run"
DATA_ROOT = REPO_ROOT / "data" / "sample" if DATA_MODE == "sample" else REAL_DATA_ROOT
ETS_OUT = RESULTS_ROOT / "ets_baseline"
TCAD_OUT = RESULTS_ROOT / "tcad_ablation"
TCAD_REPEAT_OUT = RESULTS_ROOT / "tcad_ablation_repeat"
FPGA_OUT = RESULTS_ROOT / "fpga"
RTL_SWEEP_OUT = RESULTS_ROOT / "rtl_sweep"

from exact.repro import configure_reproducibility

env_updates = configure_reproducibility(seed=SEED, threads=THREADS, matplotlib_backend="Agg")
for key, value in env_updates.items():
    print(f"{key}={value}")

PYTHONHASHSEED=123
OMP_NUM_THREADS=1
OPENBLAS_NUM_THREADS=1
MKL_NUM_THREADS=1
NUMEXPR_NUM_THREADS=1
VECLIB_MAXIMUM_THREADS=1
MPLCONFIGDIR=/var/folders/2z/b1lzf17n4z90pz80khfc42fw0000gp/T/exact-matplotlib
MPLBACKEND=Agg


## 2. Prepare Data

For the smoke run, this cell generates deterministic synthetic telemetry in the same schema used by the tests. For the journal run, replace this with an immutable real telemetry snapshot under `data/telemetry/raw/<snapshot_id>/` and `data/telemetry/processed/<snapshot_id>/`.

In [3]:
from exact.sample_data import create_sample_dataset

if DATA_MODE == "sample":
    created = create_sample_dataset(DATA_ROOT, seed=SEED, n_rows=SAMPLE_ROWS)
    print(f"Generated {len(created)} deterministic telemetry CSVs under {DATA_ROOT}")
else:
    csvs = sorted(DATA_ROOT.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {DATA_ROOT}")
    print(f"Using {len(csvs)} telemetry CSVs from {DATA_ROOT}")

sample_files = sorted(DATA_ROOT.glob("*.csv"))[:8]
display(pd.DataFrame({"example_input_files": [str(p.relative_to(REPO_ROOT)) for p in sample_files]}))

Generated 32 deterministic telemetry CSVs under /Users/hsiaopingni/Documents/New project/EXACT-TCAD/data/sample


,example_input_files
0,data/sample/DDR4_DROOP_dft.csv
1,data/sample/DDR4_DROOP_dj.csv
2,data/sample/DDR4_DROOP_mm.csv
3,data/sample/DDR4_DROOP_tr.csv
4,data/sample/DDR4_RH_dft.csv
5,data/sample/DDR4_RH_dj.csv
6,data/sample/DDR4_RH_mm.csv
7,data/sample/DDR4_RH_tr.csv


## 3. ETS-Style Baseline Reproduction

This section runs the conference-version pipeline on the selected telemetry snapshot:

1. load Setup A and Setup B telemetry
2. clean and debias telemetry
3. fit benign-only CINTAS calibration
4. build causal/ranking artifacts
5. evaluate decision-block metrics
6. write figures, CSV summaries, and a run manifest

In [4]:
from exact.experiments.ets2026 import ETS2026Config, run_ets2026

ets_cfg = ETS2026Config(
    window_sizes=(50, 100, 200),
    n_splits=3,
    lambda_res=0.5,
    agg_mode="max",
    seed=SEED,
)

ets_artifacts = run_ets2026(data_root=DATA_ROOT, out_root=ETS_OUT, cfg=ets_cfg)
print(f"ETS manifest: {ets_artifacts['run_manifest'].relative_to(REPO_ROOT)}")
print(f"Shared features: {len(ets_artifacts['shared_features'])}")

ets_a = pd.read_csv(ETS_OUT / "SETUP_A_EXACT_summary_global.csv")
ets_b = pd.read_csv(ETS_OUT / "SETUP_B_EXACT_summary_global.csv")
display(pd.concat([ets_a.assign(setup="A"), ets_b.assign(setup="B")], ignore_index=True).head(12))

ETS manifest: results/notebook_run/ets_baseline/run_manifest.json
Shared features: 12


,scenario,window_size,auc_roc,auc_pr,f1,bal_acc,mcc,brier,ece,n_test,setup
0,DROOP,50,1.000000,1.000000,0.989899,0.989583,0.979779,0.024592,0.091425,32.0,A
1,DROOP,100,1.000000,1.000000,0.980392,0.979167,0.960639,0.023117,0.087328,16.0,A
2,DROOP,200,1.000000,1.000000,0.933333,0.916667,0.859117,0.001588,0.023993,8.0,A
3,RH,50,0.489583,0.510980,0.000000,0.489583,-0.059868,0.329833,0.329990,32.0,A
4,RH,100,0.492188,0.518207,0.000000,0.479167,-0.086066,0.331873,0.348554,16.0,A
5,RH,200,0.375000,0.494444,0.000000,0.416667,-0.192450,0.355985,0.327568,8.0,A
6,SPECTRE,50,0.546875,0.565534,0.111111,0.520833,0.086066,0.310010,0.283192,32.0,A
7,SPECTRE,100,0.500000,0.574088,0.207407,0.541667,0.125988,0.326620,0.348494,16.0,A
8,SPECTRE,200,0.479167,0.634722,0.300000,0.541667,0.125988,0.307716,0.426680,8.0,A
9,DROOP,50,1.000000,1.000000,0.989899,0.989583,0.979779,0.029970,0.106610,32.0,B


## 4. TCAD Design-Space Ablation

This section runs the TCAD extension sweep. The smoke preset is intentionally small; the full preset expands feature budgets, decision-block sizes, score weights, aggregation choices, and fixed-point precisions.

Each summary row includes detection metrics and hardware-cost columns derived from the operator table under `hardware/`.

In [5]:
from exact.experiments.tcad2026 import TCAD2026Config, run_tcad_ablation

cfg_path = REPO_ROOT / "configs" / f"tcad_grid_{TCAD_PRESET}.json"
tcad_cfg = TCAD2026Config.from_json(cfg_path)
print(f"TCAD config: {cfg_path.relative_to(REPO_ROOT)}")

# Keep the notebook seed explicit even when the config file changes.
tcad_cfg = TCAD2026Config(
    scenarios_eval=tcad_cfg.scenarios_eval,
    feature_budgets=tcad_cfg.feature_budgets,
    window_sizes=tcad_cfg.window_sizes,
    lambda_res_values=tcad_cfg.lambda_res_values,
    agg_modes=tcad_cfg.agg_modes,
    weight_modes=tcad_cfg.weight_modes,
    fixed_point_q=tcad_cfg.fixed_point_q,
    n_splits=tcad_cfg.n_splits,
    p_quantile=tcad_cfg.p_quantile,
    corr_threshold=tcad_cfg.corr_threshold,
    seed=SEED,
)

tcad_artifacts = run_tcad_ablation(data_root=DATA_ROOT, out_root=TCAD_OUT, cfg=tcad_cfg)
summary = tcad_artifacts["summary"].copy()
print(f"TCAD summary: {tcad_artifacts['summary_path'].relative_to(REPO_ROOT)}")
print(f"TCAD manifest: {tcad_artifacts['manifest_path'].relative_to(REPO_ROOT)}")
display(summary.head(10))

TCAD config: configs/tcad_grid_smoke.json


TCAD summary: results/notebook_run/tcad_ablation/tcad_ablation_summary.csv
TCAD manifest: results/notebook_run/tcad_ablation/run_manifest.json


,setup,scenario,window_size,agg_mode,lambda_res,p_quantile,top_k,weight_mode,fixed_point_q,n_selected_features,...,hw_area_mm2,hw_power_mw,hw_setup_b_area_overhead_pct,hw_idle_power_overhead_pct,hw_median_workload_power_overhead_pct,hw_add_count,hw_mult_count,hw_estimated_serial_cycles,hw_add_delay_ps,hw_mult_delay_ps
0,A,DROOP,50,max,0.25,0.99,8,uniform,12,8,...,0.096727,11.5768,0.044937,0.032611,0.011960,13.0,18.0,75.0,62.7,29.09
1,A,DROOP,50,max,0.25,0.99,8,uniform,15,8,...,0.096727,11.5768,0.044937,0.032611,0.011960,13.0,18.0,75.0,62.7,29.09
2,A,DROOP,50,max,0.25,0.99,15,uniform,12,12,...,0.142306,17.1176,0.066112,0.048219,0.017683,21.0,26.0,115.0,62.7,29.09
3,A,DROOP,50,max,0.25,0.99,15,uniform,15,12,...,0.142306,17.1176,0.066112,0.048219,0.017683,21.0,26.0,115.0,62.7,29.09
4,A,DROOP,50,max,0.50,0.99,8,uniform,12,8,...,0.096727,11.5768,0.044937,0.032611,0.011960,13.0,18.0,75.0,62.7,29.09
5,A,DROOP,50,max,0.50,0.99,8,uniform,15,8,...,0.096727,11.5768,0.044937,0.032611,0.011960,13.0,18.0,75.0,62.7,29.09
6,A,DROOP,50,max,0.50,0.99,15,uniform,12,12,...,0.142306,17.1176,0.066112,0.048219,0.017683,21.0,26.0,115.0,62.7,29.09
7,A,DROOP,50,max,0.50,0.99,15,uniform,15,12,...,0.142306,17.1176,0.066112,0.048219,0.017683,21.0,26.0,115.0,62.7,29.09
8,A,DROOP,100,max,0.25,0.99,8,uniform,12,8,...,0.096727,11.5768,0.044937,0.032611,0.011960,13.0,18.0,75.0,62.7,29.09
9,A,DROOP,100,max,0.25,0.99,8,uniform,15,8,...,0.096727,11.5768,0.044937,0.032611,0.011960,13.0,18.0,75.0,62.7,29.09


## 5. Reproducibility Check

This cell reruns the TCAD ablation into a second output directory and checks that the summary table is bit-for-bit identical at the DataFrame level. This is the notebook equivalent of the repo smoke test.

In [6]:
if RUN_REPEAT_CHECK:
    repeat_artifacts = run_tcad_ablation(data_root=DATA_ROOT, out_root=TCAD_REPEAT_OUT, cfg=tcad_cfg)
    repeat_summary = repeat_artifacts["summary"].copy()
    same = summary.equals(repeat_summary)
    print(f"Repeated TCAD summary equals first run: {same}")
    if not same:
        diff_cols = [c for c in summary.columns if not summary[c].equals(repeat_summary[c])]
        raise AssertionError(f"Reproducibility check failed; differing columns: {diff_cols}")
else:
    print("RUN_REPEAT_CHECK=False, skipped repeat execution.")

Repeated TCAD summary equals first run: True


## 6. Manifest And Artifact Audit

Run manifests are the reproducibility contract. They capture the config, runtime environment, package versions, input hashes, output hashes, and git commit.

In [7]:
def load_manifest(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

ets_manifest = load_manifest(ETS_OUT / "run_manifest.json")
tcad_manifest = load_manifest(TCAD_OUT / "run_manifest.json")

manifest_overview = pd.DataFrame([
    {
        "run": "ETS baseline",
        "git_commit": ets_manifest.get("git_commit"),
        "data_files": len(ets_manifest.get("data_files", [])),
        "artifacts": len(ets_manifest.get("artifacts", [])),
        "python": ets_manifest.get("python_version"),
    },
    {
        "run": "TCAD ablation",
        "git_commit": tcad_manifest.get("git_commit"),
        "data_files": len(tcad_manifest.get("data_files", [])),
        "artifacts": len(tcad_manifest.get("artifacts", [])),
        "python": tcad_manifest.get("python_version"),
    },
])
display(manifest_overview)

display(pd.DataFrame(tcad_manifest.get("data_files", [])).head())

,run,git_commit,data_files,artifacts,python
0,ETS baseline,38ddb97b19296e115ba1364a9f0f1a847b57be1f,32,10,3.13.5
1,TCAD ablation,38ddb97b19296e115ba1364a9f0f1a847b57be1f,32,6,3.13.5


,path,sha256,size_bytes
0,data/sample/DDR4_DROOP_dft.csv,b1c6c8ea8ee2ffdc7c361071fdd52d97d02c5c9678ea6b...,134682
1,data/sample/DDR4_DROOP_dj.csv,1b7a5da92560d5490c49c0deca328413b2712e21f93fe4...,134494
2,data/sample/DDR4_DROOP_mm.csv,9dc812193f208a6b5227946de320094459b6a974cb35b4...,134615
3,data/sample/DDR4_DROOP_tr.csv,784806f79b4d0c989b07ab63a1af2b5079547863872bc0...,134526
4,data/sample/DDR4_RH_dft.csv,70a7c65a79dc14f4c256879af70cf4cc9ede07321ebef2...,134458


## 7. Hardware-Cost Summary

This section uses the hardware information you provided from Eduardo Ortega's script.

Operator source table:

- add: raw area `1165.234`, power `0.178 mW`, delay `62.7 ps`, cycles `3`
- mult: raw area `4532.164`, power `0.5146 mW`, delay `29.09 ps`, cycles `2`
- raw area is divided by `1000**2`, matching the reference script
- STD block per feature: `2 * mult + add`
- STD adder tree: `(n_features - 1) * add`
- AGG block: `2 * mult`
- power scales linearly with GHz

In [8]:
from exact.hardware import OperatorCosts, compute_tableIII_setupB, estimate_cintas_hardware_cost

costs = OperatorCosts.from_csv(REPO_ROOT / "hardware" / "cintas_operator_costs.csv")
operator_table = pd.DataFrame([
    {"operator": "add", "area_mm2": costs.add_area_mm2, "power_mw_at_1ghz": costs.add_power_mw_at_1ghz, "delay_ps": costs.add_delay_ps, "cycles": costs.add_cycles},
    {"operator": "mult", "area_mm2": costs.mult_area_mm2, "power_mw_at_1ghz": costs.mult_power_mw_at_1ghz, "delay_ps": costs.mult_delay_ps, "cycles": costs.mult_cycles},
])
display(operator_table)

display(compute_tableIII_setupB(n_features=15, frequency_ghz=1.0))

hardware_cols = [
    "setup", "scenario", "top_k", "n_selected_features", "fixed_point_q",
    "hw_area_mm2", "hw_power_mw", "hw_setup_b_area_overhead_pct",
    "hw_idle_power_overhead_pct", "hw_add_count", "hw_mult_count",
]
display(summary[hardware_cols].drop_duplicates().head(12))

,operator,area_mm2,power_mw_at_1ghz,delay_ps,cycles
0,add,0.001165,0.1780,62.70,3
1,mult,0.004532,0.5146,29.09,2


,method,n_features,area_mm2,power_mw,area_overhead_pct,idle_power_overhead_pct
0,EXACT (CINTAS),15,0.178821,21.6292,0.083076,0.060927
1,OCTANE,227,NaN,NaN,1.200000,2.600000
2,E-SCOUT,230,NaN,NaN,2.200000,1.000000


,setup,scenario,top_k,n_selected_features,fixed_point_q,hw_area_mm2,hw_power_mw,hw_setup_b_area_overhead_pct,hw_idle_power_overhead_pct,hw_add_count,hw_mult_count
0,A,DROOP,8,8,12,0.096727,11.5768,0.044937,0.032611,13.0,18.0
1,A,DROOP,8,8,15,0.096727,11.5768,0.044937,0.032611,13.0,18.0
2,A,DROOP,15,12,12,0.142306,17.1176,0.066112,0.048219,21.0,26.0
3,A,DROOP,15,12,15,0.142306,17.1176,0.066112,0.048219,21.0,26.0
16,A,RH,8,8,12,0.096727,11.5768,0.044937,0.032611,13.0,18.0
17,A,RH,8,8,15,0.096727,11.5768,0.044937,0.032611,13.0,18.0
18,A,RH,15,12,12,0.142306,17.1176,0.066112,0.048219,21.0,26.0
19,A,RH,15,12,15,0.142306,17.1176,0.066112,0.048219,21.0,26.0
32,A,SPECTRE,8,8,12,0.096727,11.5768,0.044937,0.032611,13.0,18.0
33,A,SPECTRE,8,8,15,0.096727,11.5768,0.044937,0.032611,13.0,18.0


## 8. FPGA/RTL Laptop Workflow

You can approach FPGA work on a laptop in three layers.

### Layer A: No FPGA board required

1. Use Python to export fixed-point golden vectors.
2. Simulate RTL against those vectors.
3. Debug bit-exact arithmetic and stream timing.
4. Save simulation pass/fail summaries under `results/rtl_sweep/`.

This is enough to connect RTL correctness to the notebook.

### Layer B: Open-source synthesis on laptop

1. Install a simulator such as Verilator or Icarus Verilog.
2. Install Yosys for synthesis experiments.
3. For small open FPGA targets, add the matching place-and-route tools.
4. Run synthesis scripts from the notebook or terminal.
5. Write `rtl_resource_summary.csv` with LUT/FF/DSP/BRAM, timing, cycles, and estimated energy.

This gives preliminary hardware evidence, but it depends on the target family.

### Layer C: Vendor FPGA flow

1. Pick a target board and FPGA family.
2. Use the vendor toolchain for authoritative synthesis and implementation reports.
3. On macOS laptops, vendor FPGA tools are often not native; a Linux workstation, server, or VM is usually the practical path.
4. Export the reports to CSV.
5. Let this notebook merge the CSV with the TCAD ablation table.

The notebook integration path is: Python fixed-point model -> golden vectors -> RTL simulation/synthesis -> CSV report -> merged TCAD paper table.

In [9]:
tools = []
for name, command in [
    ("verilator", ["verilator", "--version"]),
    ("iverilog", ["iverilog", "-V"]),
    ("yosys", ["yosys", "-V"]),
    ("gtkwave", ["gtkwave", "--version"]),
]:
    exe = shutil.which(command[0])
    row = {"tool": name, "available": exe is not None, "path": exe or ""}
    if exe:
        try:
            completed = subprocess.run(command, capture_output=True, text=True, timeout=10)
            first_line = (completed.stdout or completed.stderr).splitlines()[0] if (completed.stdout or completed.stderr) else ""
            row["version"] = first_line
        except Exception as exc:
            row["version"] = f"version check failed: {exc}"
    else:
        row["version"] = "not installed"
    tools.append(row)

display(pd.DataFrame(tools))

,tool,available,path,version
0,verilator,False,,not installed
1,iverilog,False,,not installed
2,yosys,False,,not installed
3,gtkwave,False,,not installed


## 9. Export Fixed-Point Golden Vectors For RTL

This cell exports a small golden-vector file from the Python fixed-point CINTAS reference. The RTL testbench should stream these feature values and compare its output score against `expected_score_q`.

In [10]:
from exact.cintas import FixedPointCINTAS, FixedPointConfig

FPGA_OUT.mkdir(parents=True, exist_ok=True)
model = ets_artifacts["model_A"]
setup_a = ets_artifacts["setup_A"]
q_format = 15
fixed = FixedPointCINTAS.from_float_model(model, FixedPointConfig(q=q_format))
subset = setup_a[model.feature_cols].head(32).copy()
score_float, e1_float, e2_float = model.score_dataframe(subset)
score_q = fixed.score_dataframe(subset)

golden = subset.copy()
golden.insert(0, "sample_index", range(len(golden)))
golden["expected_score_q"] = score_q
golden["expected_score_float"] = score_float
golden["expected_e1_float"] = e1_float
golden["expected_e2_float"] = e2_float
golden_path = FPGA_OUT / f"cintas_setupA_q{q_format}_golden_vectors.csv"
golden.to_csv(golden_path, index=False)

print(f"Golden vectors: {golden_path.relative_to(REPO_ROOT)}")
display(golden.head())

Golden vectors: results/notebook_run/fpga/cintas_setupA_q15_golden_vectors.csv


,sample_index,core_cpi,core_cycles,core_ipc,dimm_temp,dram_bw,imc_reads,l2_miss,llc_miss,pkg_power,socket_temp,uops_retired,vcc_voltage,expected_score_q,expected_score_float,expected_e1_float,expected_e2_float
0,0,-0.001895,0.150320,-0.030784,-0.096411,-0.193064,-0.005325,-0.013893,0.021154,-2.060056,-0.811914,-0.222192,-0.004031,70362,2.150039,1.622265,2.677812
1,1,0.035854,0.127987,-0.017357,0.171875,-0.175753,-0.030701,-0.021205,0.014482,-1.402735,-0.595545,-0.266819,-0.001336,14678,0.445067,0.561498,0.328637
2,2,0.040946,0.178039,0.016757,0.197953,-0.136640,-0.002127,-0.020227,0.022705,-0.932496,-1.183923,-0.124313,-0.002961,44098,1.346614,1.212765,1.480463
3,3,0.048668,0.175315,-0.004123,0.086268,-0.141813,-0.010069,0.011551,0.032554,-0.528162,-0.925894,-0.253915,-0.001083,11754,0.364528,0.470883,0.258174
4,4,0.024991,0.133934,0.011401,0.362325,-0.143572,-0.016076,0.009747,0.009325,-1.355040,-0.617913,-0.288414,-0.003491,55764,1.711045,1.409854,2.012235


## 10. Merge Future RTL/FPGA Results Into The TCAD Table

When simulation or synthesis is ready, save a CSV at `results/notebook_run/rtl_sweep/rtl_resource_summary.csv` or `results/rtl_sweep/rtl_resource_summary.csv` with columns like:

```text
setup,top_k,fixed_point_q,luts,ffs,dsps,brams,fmax_mhz,latency_cycles,energy_per_block_nj
```

The cell below loads that file if it exists. Otherwise it writes a template so the expected schema is clear.

In [11]:
rtl_summary_candidates = [
    RTL_SWEEP_OUT / "rtl_resource_summary.csv",
    REPO_ROOT / "results" / "rtl_sweep" / "rtl_resource_summary.csv",
]
existing = next((p for p in rtl_summary_candidates if p.exists()), None)

if existing is None:
    RTL_SWEEP_OUT.mkdir(parents=True, exist_ok=True)
    template = pd.DataFrame([
        {
            "setup": "A",
            "top_k": int(summary["top_k"].iloc[0]),
            "fixed_point_q": int(summary["fixed_point_q"].iloc[0]),
            "luts": None,
            "ffs": None,
            "dsps": None,
            "brams": None,
            "fmax_mhz": None,
            "latency_cycles": None,
            "energy_per_block_nj": None,
            "status": "fill after RTL simulation/synthesis",
        }
    ])
    existing = RTL_SWEEP_OUT / "rtl_resource_summary.csv"
    template.to_csv(existing, index=False)
    print(f"Created RTL summary template: {existing.relative_to(REPO_ROOT)}")

rtl_summary = pd.read_csv(existing)
print(f"RTL summary source: {existing.relative_to(REPO_ROOT)}")
display(rtl_summary.head())

merge_keys = [key for key in ["setup", "top_k", "fixed_point_q"] if key in rtl_summary.columns and key in summary.columns]
if merge_keys:
    merged = summary.merge(rtl_summary, on=merge_keys, how="left")
    display(merged.head())
else:
    print("RTL summary does not yet share merge keys with the TCAD summary.")

Created RTL summary template: results/notebook_run/rtl_sweep/rtl_resource_summary.csv
RTL summary source: results/notebook_run/rtl_sweep/rtl_resource_summary.csv


,setup,top_k,fixed_point_q,luts,ffs,dsps,brams,fmax_mhz,latency_cycles,energy_per_block_nj,status
0,A,8,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fill after RTL simulation/synthesis


,setup,scenario,window_size,agg_mode,lambda_res,p_quantile,top_k,weight_mode,fixed_point_q,n_selected_features,...,hw_add_delay_ps,hw_mult_delay_ps,luts,ffs,dsps,brams,fmax_mhz,latency_cycles,energy_per_block_nj,status
0,A,DROOP,50,max,0.25,0.99,8,uniform,12,8,...,62.7,29.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fill after RTL simulation/synthesis
1,A,DROOP,50,max,0.25,0.99,8,uniform,15,8,...,62.7,29.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A,DROOP,50,max,0.25,0.99,15,uniform,12,12,...,62.7,29.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A,DROOP,50,max,0.25,0.99,15,uniform,15,12,...,62.7,29.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A,DROOP,50,max,0.50,0.99,8,uniform,12,8,...,62.7,29.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fill after RTL simulation/synthesis


## 11. Paper-Ready Output Checklist

After this notebook completes, use these generated artifacts to keep the journal work organized:

- ETS baseline manifest: `results/notebook_run/ets_baseline/run_manifest.json`
- TCAD ablation manifest: `results/notebook_run/tcad_ablation/run_manifest.json`
- TCAD ablation summary: `results/notebook_run/tcad_ablation/tcad_ablation_summary.csv`
- selected features and COM/MEM/SEN counts: `results/notebook_run/tcad_ablation/tcad_selected_features.csv`
- fixed-point golden vectors: `results/notebook_run/fpga/cintas_setupA_q15_golden_vectors.csv`
- RTL/FPGA result template or merged report: `results/notebook_run/rtl_sweep/rtl_resource_summary.csv`

For the final journal run, switch from deterministic sample data to the frozen real telemetry snapshot and run the full TCAD grid.

In [12]:
final_artifacts = [
    ETS_OUT / "run_manifest.json",
    TCAD_OUT / "run_manifest.json",
    TCAD_OUT / "tcad_ablation_summary.csv",
    TCAD_OUT / "tcad_selected_features.csv",
    golden_path,
    existing,
]

for artifact in final_artifacts:
    print(f"{'OK' if artifact.exists() else 'MISSING'}  {artifact.relative_to(REPO_ROOT)}")

print("\nNotebook run complete.")

OK  results/notebook_run/ets_baseline/run_manifest.json
OK  results/notebook_run/tcad_ablation/run_manifest.json
OK  results/notebook_run/tcad_ablation/tcad_ablation_summary.csv
OK  results/notebook_run/tcad_ablation/tcad_selected_features.csv
OK  results/notebook_run/fpga/cintas_setupA_q15_golden_vectors.csv
OK  results/notebook_run/rtl_sweep/rtl_resource_summary.csv

Notebook run complete.
